# Classifier Training

Train a 1D CNN classifier on autoencoder residuals for multi-class fault classification.

## Prerequisites

```bash
pip install -e ".[notebooks]"
```

Requires a trained autoencoder checkpoint at `models/autoencoder_best.pth`.

## Setup and imports

In [ ]:
import torch
import numpy as np

from sat_anomaly.config import get_classifier_config, validate_config
from sat_anomaly.models.classifier import CNNClassifier
from sat_anomaly.models.training import (
    train_classifier,
    load_pretrained_autoencoder,
    compute_residual_windows_batched,
)
from sat_anomaly.visualization.plots import plot_training_history
from sat_anomaly.data.loader import load_fault_data_with_annotations, create_labeled_data_loaders
from sat_anomaly.data.preprocessor import (
    normalize_features,
    create_grouped_time_windows_with_labels,
    split_sequences_with_labels,
)

print("Import successful")

## Build config

In [ ]:
config = get_classifier_config()
assert validate_config(config)
config

## Load and preprocess labeled data

In [ ]:
data, label_map = load_fault_data_with_annotations(config['data_path'])

normalized_data, _scaler = normalize_features(data)

_numeric_cols = normalized_data.select_dtypes(include=['number']).columns
feature_names = [c for c in _numeric_cols if c not in ['time_ns', 'time_s', 'label_any_fault']]
config['feature_names'] = feature_names

windows, labels = create_grouped_time_windows_with_labels(
    normalized_data,
    group_cols=['sequence_id'],
    window_size=config['window_size'],
    step_size=config['step_size'],
    label_col='fault_label_name'
)

train_windows, val_windows, train_labels, val_labels = split_sequences_with_labels(
    windows, labels, train_ratio=config['train_ratio']
)

observed = sorted(set(train_labels.tolist() + val_labels.tolist()))
label_to_id = {name: idx for idx, name in enumerate(observed)}
config['n_classes'] = len(label_to_id)

train_id_labels = np.array([label_to_id[x] for x in train_labels], dtype=np.int64)
val_id_labels = np.array([label_to_id[x] for x in val_labels], dtype=np.int64)

## Compute residuals and create data loaders

In [ ]:
ae_model = load_pretrained_autoencoder(config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bs = int(config.get('residual_batch_size', max(16, config.get('batch_size', 64))))

train_windows = compute_residual_windows_batched(ae_model, train_windows, device=device, batch_size=bs)
val_windows = compute_residual_windows_batched(ae_model, val_windows, device=device, batch_size=bs)

train_loader, val_loader = create_labeled_data_loaders(
    train_windows, train_id_labels,
    val_windows, val_id_labels,
    config['batch_size']
)

## Create model and train

In [ ]:
model = CNNClassifier(
    n_features=config['n_features'],
    n_classes=config['n_classes'],
    channels=config.get('cnn_channels', [64, 128, 256]),
    dropout=config.get('dropout', 0.2)
)
model = model.to(device)
print(f'Using device: {device}')

train_results = train_classifier(model, train_loader, val_loader, config)
print('Training completed!')
plot_training_history(train_results['train_losses'], train_results['val_losses'])

## Visualize residuals

In [ ]:
import random
import math
import matplotlib.pyplot as plt

idx = random.randrange(len(val_windows))
residual_seq = val_windows[idx]
label_id = int(val_id_labels[idx])
id_to_label = {v: k for k, v in label_to_id.items()}
label_name = id_to_label.get(label_id, f'label_id={label_id}')

T, F = residual_seq.shape
ncols = 4
nrows = math.ceil(F / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.2 * nrows), sharex=True)
axes = axes.ravel()

for fi in range(F):
    ax = axes[fi]
    name = feature_names[fi] if fi < len(feature_names) else f'Feature {fi}'
    ax.plot(range(T), residual_seq[:, fi], label='Residual', color='C3', linewidth=1.0)
    ax.set_title(name, fontsize=9)
    ax.grid(True, alpha=0.3)

for j in range(F, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Validation residual window (label: {label_name})')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()